# UC1 — AI Data Quality Reporter

**Purpose:** Read rejected records from the Silver quarantine tables, group them by issue type, generate a plain-English audit explanation for each group using an LLM, and write the results to the Gold layer.

**Output table:** `globalmart.gold.dq_audit_report`

**Run order:**
1. Install dependencies (run once, then restart Python)
2. Setup — imports, config, LLM connection
3. Load quarantine tables
4. Watermark — skip records already processed on a previous run
5. Aggregate — group rejections by issue type
6. Classify severity per issue group
7. Build prompt and system message
8. Generate AI explanations
9. Write results to Gold layer
10. Log the run

---

## Step 1 — Install dependencies

Run this cell once, then restart Python before continuing.

In [0]:
%pip install openai --quiet
dbutils.library.restartPython()

## Step 2 — Imports and configuration

In [0]:
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from openai import OpenAI

# ── LLM connection ────────────────────────────────────────────────────────
# Uses the workspace token automatically — no API key needed
client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url="https://4092199907892374.ai-gateway.cloud.databricks.com/mlflow/v1/"
)
MODEL_NAME    = "databricks-gpt-oss-20b"
NOTEBOOK_NAME = "UC1_AI_Data_Quality_Reporter"

# ── Quarantine tables (Silver layer rejects land here) ────────────────────
TABLE_CUSTOMERS    = "globalmart.silver.customers_quarantine"
TABLE_ORDERS       = "globalmart.silver.orders_quarantine"
TABLE_TRANSACTIONS = "globalmart.silver.transactions_quarantine"
TABLE_PRODUCTS     = "globalmart.silver.products_quarantine"
TABLE_RETURNS      = "globalmart.silver.returns_quarantine"
TABLE_VENDORS      = "globalmart.silver.vendors_quarantine"

# ── Output tables (Gold layer) ────────────────────────────────────────────
TABLE_AUDIT   = "globalmart.gold.dq_audit_report"
TABLE_RUN_LOG = "globalmart.gold.pipeline_run_log"

print(f"Setup complete | Model: {MODEL_NAME} | Started: {datetime.now():%Y-%m-%d %H:%M:%S}")

## Step 3 — LLM helper functions

In [0]:
def extract_text(response) -> str:
    """
    Pull the answer text out of the model response.

    databricks-gpt-oss-20b returns a list of content blocks:
      [{"type": "reasoning", ...}, {"type": "text", "text": "the answer"}]
    We only want the "text" block. If the response is already a plain
    string (older model versions), return it directly.
    """
    content = response.choices[0].message.content
    if isinstance(content, list):
        return " ".join(
            block["text"]
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        ).strip()
    return content.strip()


def call_llm(prompt: str, system_msg: str, retries: int = 3, wait: int = 2) -> str:
    """
    Call the LLM with automatic retry on failure.

    Returns the answer text on success, or a string starting with
    'LLM_UNAVAILABLE:' if all retries fail — so the pipeline keeps
    running and failures are recorded rather than crashing.
    """
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.3   # Low temperature = consistent, factual output
            )
            return extract_text(response)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(wait * (attempt + 1))  # Back off before retrying
            else:
                return f"LLM_UNAVAILABLE: {e}"


def validate_llm_output(text: str, min_words: int = 30) -> dict:
    """
    Check the LLM output quality before writing to the audit table.

    Returns a dict with:
      text         -- the explanation text (unchanged)
      llm_check -- PASS or REVIEW
      issues       -- comma-separated list of problems found, or 'none'

    REVIEW means a human should check the explanation before sharing
    it with the external auditor.
    """
    # Phrases that indicate the model refused to answer
    refusal_phrases = ["i cannot", "as an ai", "i don't have", "i'm unable", "i am unable"]

    # Technical terms that should not appear in a finance-facing explanation
    jargon_phrases  = ["silver layer", "bronze layer", "harmonization process",
                       "the analytics layer", "the pipeline", "dlt", "delta table"]

    issues = []

    if len(text.split()) < min_words:
        issues.append("too_short")

    if text.startswith("LLM_UNAVAILABLE"):
        issues.append("llm_call_failed")

    if any(phrase in text.lower() for phrase in refusal_phrases):
        issues.append("refusal_detected")

    if any(phrase in text.lower() for phrase in jargon_phrases):
        issues.append("technical_jargon_detected")

    # Strip markdown formatting the model occasionally adds despite instructions
    import re
    text = re.sub(r'\*{1,2}([^*]+)\*{1,2}', r'\1', text)  # remove **bold** and *italic*
    text = re.sub(r'#{1,3}\s*', '', text)                    # remove ## headers
    text = text.strip()

    return {
        "text":         text,
        "llm_check": "PASS" if not issues else "REVIEW",
        "issues":       ", ".join(issues) if issues else "none"
    }


print("LLM helpers ready.")

## Step 4 — Load quarantine tables

Each table holds records that the Silver pipeline rejected. All six tables share the same metadata columns:

| Column | What it contains |
|---|---|
| `_source_file` | Exact path of the CSV/JSON the record came from |
| `_expectation_name` | Name of the quality rule that rejected the record |
| `_issue_type` | Category: `missing_optional_field`, `format_error`, `invalid_category` |
| `_failure_reason` | The actual failure message, including the bad value |
| `_quarantine_timestamp` | When the record was rejected |

In [0]:
df_q_customers    = spark.table(TABLE_CUSTOMERS)
df_q_orders       = spark.table(TABLE_ORDERS)
df_q_transactions = spark.table(TABLE_TRANSACTIONS)
df_q_products     = spark.table(TABLE_PRODUCTS)
df_q_returns      = spark.table(TABLE_RETURNS)
df_q_vendors      = spark.table(TABLE_VENDORS)

print("Quarantine table row counts:")
for name, df in [
    ("customers",    df_q_customers),
    ("orders",       df_q_orders),
    ("transactions", df_q_transactions),
    ("products",     df_q_products),
    ("returns",      df_q_returns),
    ("vendors",      df_q_vendors),
]:
    print(f"  {name:<15} : {df.count():,} rejected records")

## Step 5 — Incremental watermark

On the first run, all quarantine records are processed.
On subsequent runs, only records quarantined *after* the last run are processed.
This prevents duplicate explanations being generated for the same issue group.

In [0]:
# Find the timestamp of the last successful run
try:
    last_run_ts = spark.sql(
        f"SELECT MAX(generated_at) AS last_ts FROM {TABLE_AUDIT}"
    ).collect()[0]["last_ts"]
    print(f"Last run: {last_run_ts} — processing new records only.")
except Exception:
    last_run_ts = None
    print("No previous run found — processing all quarantine records.")


def apply_watermark(df):
    """Keep only records quarantined after the last run. Skip filter if first run."""
    if last_run_ts is not None and "_quarantine_timestamp" in df.columns:
        return df.filter(F.col("_quarantine_timestamp") > F.lit(last_run_ts))
    return df


df_q_customers    = apply_watermark(df_q_customers)
df_q_orders       = apply_watermark(df_q_orders)
df_q_transactions = apply_watermark(df_q_transactions)
df_q_products     = apply_watermark(df_q_products)
df_q_returns      = apply_watermark(df_q_returns)
df_q_vendors      = apply_watermark(df_q_vendors)

print("\nPost-watermark counts (records to process this run):")
for name, df in [
    ("customers",    df_q_customers),
    ("orders",       df_q_orders),
    ("transactions", df_q_transactions),
    ("products",     df_q_products),
    ("returns",      df_q_returns),
    ("vendors",      df_q_vendors),
]:
    print(f"  {name:<15} : {df.count():,}")

## Step 6 — Aggregate rejections into issue groups

Instead of generating one explanation per rejected row (slow and repetitive), we group by rule name and issue type first. One LLM call per group covers all records that failed for the same reason.

For each group we collect:
- **Sample failure messages** from `_failure_reason` — actual bad values the LLM can cite
- **Source files** from `_source_file` — which regional CSV/JSON files contributed the failures
- **Time range** — when the earliest and latest failures occurred

In [0]:
def aggregate_quarantine(df, entity_name: str, table_name: str):
    """
    Group rejections by rule name and issue type.
    Collect evidence so the LLM prompt has concrete data to reference.
    """
    return (
        df.groupBy("_issue_type", "_expectation_name")
          .agg(
              F.count("*").alias("rejected_count"),
              # Up to 3 sample failure messages — gives the LLM real examples of bad data
              F.slice(F.collect_set("_failure_reason"), 1, 3).alias("sample_reasons"),
              # Up to 5 source file paths — shows which regional files are affected
              F.slice(F.collect_set("_source_file"), 1, 5).alias("source_files"),
              F.min("_quarantine_timestamp").alias("earliest_failure"),
              F.max("_quarantine_timestamp").alias("latest_failure"),
          )
          .withColumn("entity",       F.lit(entity_name))
          .withColumn("source_table", F.lit(table_name))
          .withColumnRenamed("_expectation_name", "field_affected")
          .withColumnRenamed("_issue_type",        "issue_type")
    )


# Union all six entities into one dataframe
dq_df = (
    aggregate_quarantine(df_q_customers,    "customers",    TABLE_CUSTOMERS)
    .unionByName(aggregate_quarantine(df_q_orders,       "orders",       TABLE_ORDERS),       allowMissingColumns=True)
    .unionByName(aggregate_quarantine(df_q_transactions, "transactions", TABLE_TRANSACTIONS), allowMissingColumns=True)
    .unionByName(aggregate_quarantine(df_q_products,     "products",     TABLE_PRODUCTS),     allowMissingColumns=True)
    .unionByName(aggregate_quarantine(df_q_returns,      "returns",      TABLE_RETURNS),      allowMissingColumns=True)
    .unionByName(aggregate_quarantine(df_q_vendors,      "vendors",      TABLE_VENDORS),      allowMissingColumns=True)
    .filter(F.col("rejected_count") > 0)
)

# Collect the full dataframe — all columns needed by the generation loop
dq_summary = dq_df.collect()

print(f"Issue groups to explain this run: {len(dq_summary)}")

# Display a slim view for quick review
display(
    dq_df
    .select("entity", "field_affected", "issue_type", "rejected_count", "source_table")
    .orderBy("entity", F.col("rejected_count").desc())
)

## Step 7 — Classify severity

Each issue group is classified so the finance team knows which rows to address first.

| Severity | Criteria |
|---|---|
| CRITICAL | Missing primary key, null ID, invalid category on a financial field, or > 100 records |
| HIGH | Invalid reference, negative amount, format error, or > 30 records |
| MEDIUM | Missing optional fields, invalid ranges, everything else |

In [0]:
def classify_severity(entity: str, issue_type: str, count: int) -> str:
    critical_issues = {
        "missing_value", "null_primary_key",
        "missing_customer_id", "missing_order_id",
        "invalid_category"
    }
    high_issues = {
        "invalid_reference", "negative_amount",
        "orphaned_record", "duplicate_primary_key",
        "format_error"
    }

    if issue_type.lower() in critical_issues or count > 100:
        return "CRITICAL"
    elif issue_type.lower() in high_issues or count > 30:
        return "HIGH"
    else:
        return "MEDIUM"


print("Severity classifier ready.")

## Step 8 — Build the LLM prompt

The prompt produces a **three-section structured explanation** so a finance director can scan it in seconds:

- **WHAT HAPPENED** — exact field, exact count, what the bad data looked like
- **WHERE IT CAME FROM** — source files and regional systems
- **BUSINESS RISK AND ACTION** — which report breaks and exactly what to do

Key design decisions:
- Real table names are used (`globalmart.silver.orders_quarantine`), not vague labels
- Sample failure messages are injected so the LLM can cite actual bad values
- Source file paths are shortened to `filename.csv (Region N)` for readability
- Technical terms (Silver layer, pipeline, ETL) are explicitly banned in the rules block

In [0]:
def _format_sample_reasons(sample_reasons: list) -> str:
    """Format sample failure messages for the prompt."""
    if not sample_reasons:
        return "  - No sample messages available"
    return "\n".join(f"  - {r}" for r in sample_reasons[:3])


def _format_source_files(source_files: list) -> str:
    """
    Shorten full Volume paths to readable filenames with region labels.
    /Volumes/globalmart/raw/.../Region 3/customers_3.csv
    becomes: customers_3.csv (Region 3)
    """
    if not source_files:
        return "  - Source file information not available"
    lines = []
    for path in list(source_files)[:4]:
        filename = path.split("/")[-1] if "/" in path else path
        region   = ""
        for part in path.split("/"):
            if "Region" in part or "region" in part:
                region = f" ({part})"
                break
        lines.append(f"  - {filename}{region}")
    return "\n".join(lines)


def build_dq_prompt(
    entity: str, field: str, issue_type: str, count: int, severity: str,
    source_table: str, sample_reasons: list, source_files: list,
    earliest_failure: str, latest_failure: str,
) -> str:
    """Build the prompt for one issue group."""

    # Time range note — used only if failures span multiple days
    time_note = ""
    if earliest_failure and latest_failure and earliest_failure != latest_failure:
        time_note = f"Failures span {earliest_failure[:10]} to {latest_failure[:10]}."
    elif earliest_failure:
        time_note = f"First recorded on {earliest_failure[:10]}."

    reasons_block = _format_sample_reasons(sample_reasons)
    files_block   = _format_source_files(source_files)

    return (
        "You are writing a data quality audit report for GlobalMart's external finance auditors.\n"
        "GlobalMart unified data from 6 disconnected regional systems. "
        "Records that failed validation were set aside before reaching any reports or financial figures.\n\n"
        "Write the explanation in EXACTLY this format — three labelled sections, each one or two sentences only:\n\n"
        f"WHAT HAPPENED: [Name the exact field ({field}), exact count ({count:,} records), and describe "
        "what was wrong using the sample messages below. "
        "Do not say 'was flagged' — say what the data actually looked like.]\n\n"
        "WHERE IT CAME FROM: [Name the specific source files and regions from the list below. "
        "Start with 'These records came from...']\n\n"
        "BUSINESS RISK AND ACTION: [Name which specific report or figure is broken. "
        "Then state who must do what to fix it before the audit deadline.]\n\n"
        f"Evidence:\n"
        f"  Table    : {source_table}\n"
        f"  Severity : {severity}\n"
        f"  {time_note}\n\n"
        f"  Sample failure messages:\n{reasons_block}\n\n"
        f"  Source files:\n{files_block}\n\n"
        "Rules:\n"
        "- Do NOT use: Silver layer, Bronze layer, pipeline, DLT, Spark, Delta, ETL, quarantine table, harmonization\n"
        "- Instead say: 'removed from the reporting database', 'set aside before processing'\n"
        f"- Always name the field ({field}) and count ({count:,}) in the first section\n"
        "- Always name at least one source file in the second section\n"
        "- Third section must name a specific report (revenue total, profit calculation, audit trail)\n"
        "- One or two sentences per section — the reader is scanning, not reading\n"
        "- Start directly with WHAT HAPPENED — no preamble"
    )


SYSTEM_MSG_UC1 = (
    "You are a senior data quality analyst writing audit documentation for a finance director. "
    "You must answer exactly three questions in order, each labelled exactly as shown. "
    "Write in plain English — name specific fields, counts, files, and reports. "
    "Never use database or programming terminology. "
    "Do not use any markdown formatting — no asterisks, no bold, no italic, no bullet points. "
    "Plain text only."
)

print("Prompt builder ready.")

## Step 9 — Generate AI explanations

One LLM call per issue group. Results are collected into `audit_records` and printed so you can review quality before writing to the table.

In [0]:
audit_records  = []
llm_call_count = 0
review_count   = 0

for row in dq_summary:
    # Unpack the aggregated row
    entity         = row["entity"]
    field          = row["field_affected"]
    issue          = row["issue_type"]
    count          = row["rejected_count"]
    source_table   = row["source_table"]
    sample_reasons = list(row["sample_reasons"]) if row["sample_reasons"] else []
    source_files   = list(row["source_files"])   if row["source_files"]   else []
    earliest       = str(row["earliest_failure"]) if row["earliest_failure"] else ""
    latest         = str(row["latest_failure"])   if row["latest_failure"]   else ""
    severity       = classify_severity(entity, issue, count)

    # Generate the explanation
    prompt     = build_dq_prompt(entity, field, issue, count, severity,
                                  source_table, sample_reasons, source_files, earliest, latest)
    raw_output = call_llm(prompt, SYSTEM_MSG_UC1)
    validated  = validate_llm_output(raw_output)
    llm_call_count += 1

    if validated["llm_check"] == "REVIEW":
        review_count += 1

    # Build the record to write to the audit table
    audit_records.append({
        "entity":           entity,
        "field_affected":   field,
        "issue_type":       issue,
        "rejected_count":   int(count),
        "severity":         severity,
        "source_table":     source_table,
        "source_files":     " | ".join(source_files[:5]) if source_files else "not available",
        "earliest_failure": earliest,
        "latest_failure":   latest,
        "sample_evidence":  " | ".join(sample_reasons[:3]) if sample_reasons else "none",
        "ai_explanation":   validated["text"],
        "llm_check":     validated["llm_check"],
        "llm_check_detail":   validated["issues"],
        "generated_at":     datetime.now().isoformat()
    })

    # Print a review summary
    print("=" * 70)
    print(f"[{severity}] {entity} | {field} | {issue} | {count:,} records")
    print(f"Source : {source_table}")
    print(f"Files  : {', '.join(source_files[:2]) if source_files else 'unknown'}")
    if validated["llm_check"] == "REVIEW":
        print(f"*** REVIEW FLAG: {validated['issues']} ***")
    print(f"\n{validated['text']}")
    print("=" * 70 + "\n")

print(f"Done. {len(audit_records)} explanations generated.")
print(f"LLM calls: {llm_call_count}  |  Flagged for review: {review_count}")

## Step 10 — Write audit report to Gold layer

Appends to `globalmart.gold.dq_audit_report`. `mergeSchema: true` handles new columns being added to an existing table automatically.

In [0]:
schema = StructType([
    StructField("entity",           StringType(),  True),
    StructField("field_affected",   StringType(),  True),
    StructField("issue_type",       StringType(),  True),
    StructField("rejected_count",   IntegerType(), True),
    StructField("severity",         StringType(),  True),
    StructField("source_table",     StringType(),  True),
    StructField("source_files",     StringType(),  True),
    StructField("earliest_failure", StringType(),  True),
    StructField("latest_failure",   StringType(),  True),
    StructField("sample_evidence",  StringType(),  True),
    StructField("ai_explanation",   StringType(),  True),
    StructField("llm_check",     StringType(),  True),
    StructField("llm_check_detail",   StringType(),  True),
    StructField("generated_at",     StringType(),  True),
])

df_audit = spark.createDataFrame(audit_records, schema=schema)

(
    df_audit.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(TABLE_AUDIT)
)

print(f"Written to {TABLE_AUDIT}")
print(f"Rows written this run: {df_audit.count()}")

# Display sorted by severity then volume
display(
    spark.table(TABLE_AUDIT)
    .orderBy(
        F.when(F.col("severity") == "CRITICAL", 1)
         .when(F.col("severity") == "HIGH",     2)
         .otherwise(3),
        F.col("rejected_count").desc()
    )
)

## Step 11 — Log the run

Records pipeline metadata for the Pipeline Health dashboard.

In [0]:
run_log = [{
    "notebook":          NOTEBOOK_NAME,
    "run_timestamp":     datetime.now().isoformat(),
    "records_processed": len(dq_summary),
    "llm_calls_made":    llm_call_count,
    "outputs_flagged":   review_count,
    "status":            "SUCCESS" if review_count < llm_call_count else "PARTIAL_REVIEW",
    "notes":             f"watermark_applied={last_run_ts is not None}"
}]

(
    spark.createDataFrame(run_log)
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(TABLE_RUN_LOG)
)

print(f"Run log written to {TABLE_RUN_LOG}")
print(f"Status         : {run_log[0]['status']}")
print(f"Issue groups   : {len(dq_summary)}")
print(f"LLM calls      : {llm_call_count}")
print(f"Flagged REVIEW : {review_count}")